# 02d — Training Deformable DETR (R50 / R101)

**Runtime:** V100 16GB. T4 feasible untuk R50 dengan batch=2.

**PERINGATAN — Setup paling kompleks di seluruh project ini.**

Deformable DETR membutuhkan kompilasi CUDA operator (`MultiScaleDeformableAttention`) dari source. Ini proses satu kali yang memakan ~10 menit, tapi wajib dilakukan sebelum training.

**Yang membedakan DETR dari model lain:**
- **Set-prediction**: tidak ada NMS, tidak ada anchor box
- **Hungarian matching loss**: model belajar assign setiap query ke satu instance
- **Fixed queries**: jumlah query = 1 (soccer field = always single instance)
- **End-to-end**: deteksi + pose dalam satu forward pass

| Variant | GPU | Estimasi waktu |
|---|---|---|
| R50  | V100 16GB | ~8–10 jam |
| R101 | V100 16GB | ~12–15 jam |

> ⚠️ Jika CUDA ops gagal dikompilasi, jalankan Cell 3 di runtime baru (Runtime → Disconnect and delete runtime).

In [ ]:
# Cell 1: Install base dependencies
!pip install torch==2.1.0 torchvision==0.16.0 --quiet
!pip install timm einops scipy --quiet
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB' if torch.cuda.is_available() else '')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Environment & Path Setup — kompatibel Kaggle DAN Colab (auto-detect)
# ══════════════════════════════════════════════════════════════════════
import os, sys

def _detect_env():
    if os.path.exists('/kaggle/working'):
        return 'kaggle'
    if os.path.exists('/content'):
        return 'colab'
    return 'local'

ENV = _detect_env()
print(f'Environment terdeteksi: {ENV}')

# ⚠️ GANTI dengan URL repo GitHub kamu sendiri (isi SEKALI saja per notebook)
GITHUB_REPO = 'https://github.com/atangorp/soccer-homography-benchmark.git'

def _link(src, dst):
    """Symlink src -> dst. Aman dipanggil berkali-kali (idempotent),
    termasuk kalau dst adalah broken symlink dari sesi sebelumnya."""
    if os.path.exists(src) and not os.path.lexists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        os.symlink(src, dst)

if ENV == 'kaggle':
    PROJECT_ROOT = '/kaggle/working/soccer-homography-benchmark'
    if not os.path.exists(PROJECT_ROOT):
        os.system(f'git clone -q {GITHUB_REPO} "{PROJECT_ROOT}"')
    if not os.path.exists(f'{PROJECT_ROOT}/src'):
        raise RuntimeError(
            f'Git clone gagal atau repo kosong: {GITHUB_REPO}\n'
            f'Cek: (1) URL repo benar, (2) repo sudah di-push berisi folder src/,\n'
            f'(3) repo Public atau token akses sudah diset via Kaggle Secrets.'
        )
    sys.path.insert(0, PROJECT_ROOT)

    # WAJIB: attach dataset "soccer-field-raw" via "+ Add Input" di sidebar kanan
    # (ganti 'soccer-field-raw' kalau slug dataset kamu berbeda)
    _link('/kaggle/input/soccer-field-raw',
          f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8')
    # Output notebook 01 (COCO JSON) — attach "01-dataset-conversion" via "+ Add Input" -> Notebook Output
    _link(f'/kaggle/input/01-dataset-conversion/soccer-homography-benchmark/data/converted/annotations',
          f'{PROJECT_ROOT}/data/converted/annotations')
    # Symlink gambar asli ke struktur data/converted/images/{split} (dibutuhkan training MMPose)
    for _split in ['train', 'val', 'test']:
        _link(f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8/{_split}/images',
              f'{PROJECT_ROOT}/data/converted/images/{_split}')

elif ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Colab Notebooks/soccer-homography-benchmark'
    sys.path.insert(0, PROJECT_ROOT)

else:
    PROJECT_ROOT = os.getcwd()
    sys.path.insert(0, PROJECT_ROOT)

COCO_ANN_DIR = f'{PROJECT_ROOT}/data/converted/annotations'
IMAGES_DIR   = f'{PROJECT_ROOT}/data/converted/images'
for split in ['train', 'val', 'test']:
    p = f'{COCO_ANN_DIR}/{split}.json'
    print(f'  {split}.json: {"OK" if os.path.exists(p) else "TIDAK ADA"}')
print(f'PROJECT_ROOT : {PROJECT_ROOT}')


In [ ]:
# ── Pastikan semua folder output ada (wajib di Kaggle) ───────────────────────
import os
_dirs_to_create = [
    f'{PROJECT_ROOT}/artifacts/results/figures',
    f'{PROJECT_ROOT}/artifacts/results/tables',
    f'{PROJECT_ROOT}/artifacts/results/videos',
    f'{PROJECT_ROOT}/artifacts/logs/evaluation',
    f'{PROJECT_ROOT}/artifacts/logs/training',
    f'{PROJECT_ROOT}/artifacts/weights/yolo11/small',
    f'{PROJECT_ROOT}/artifacts/weights/yolo11/medium',
    f'{PROJECT_ROOT}/artifacts/weights/yolo11/xlarge',
    f'{PROJECT_ROOT}/artifacts/weights/hrnet/w18',
    f'{PROJECT_ROOT}/artifacts/weights/hrnet/w32',
    f'{PROJECT_ROOT}/artifacts/weights/hrnet/w48',
    f'{PROJECT_ROOT}/artifacts/weights/vitpose/small',
    f'{PROJECT_ROOT}/artifacts/weights/vitpose/base',
    f'{PROJECT_ROOT}/artifacts/weights/vitpose/large',
    f'{PROJECT_ROOT}/artifacts/weights/detr/r50',
    f'{PROJECT_ROOT}/artifacts/weights/detr/r101',
    f'{PROJECT_ROOT}/data/converted/annotations',
    f'{PROJECT_ROOT}/data/converted/images/train',
    f'{PROJECT_ROOT}/data/converted/images/val',
    f'{PROJECT_ROOT}/data/converted/images/test',
]
for _d in _dirs_to_create:
    os.makedirs(_d, exist_ok=True)
print('Semua folder output siap')

In [ ]:
# Cell 3: Clone dan compile Deformable DETR CUDA ops
# ⚠️ Ini yang paling sering error. Jika gagal, lihat troubleshooting di Cell 4.
import os

DETR_DIR = '/content/deformable_detr'

if not os.path.exists(DETR_DIR):
    print('Cloning Deformable-DETR...')
    !git clone https://github.com/fundamentalvision/Deformable-DETR.git {DETR_DIR}
    print('Done')
else:
    print('Repo sudah ada')

# Kompilasi CUDA operator MultiScaleDeformableAttention
# Ini wajib dilakukan sebelum import apapun dari DETR
print('\nCompiling CUDA ops (estimasi ~10 menit)...')
os.chdir(f'{DETR_DIR}/models/ops')
!python setup.py build install 2>&1 | tail -5
os.chdir('/content')

# Verifikasi compile berhasil
print('\nVerifying CUDA ops...')
sys.path.insert(0, DETR_DIR)
try:
    from models.ops.functions import MSDeformAttnFunction
    print('MultiScaleDeformableAttention: OK')
except Exception as e:
    print(f'GAGAL: {e}')
    print('Coba: Runtime → Disconnect and delete runtime, lalu jalankan ulang dari Cell 1')

In [ ]:
# Cell 4: Troubleshooting CUDA ops (jalankan ini jika Cell 3 gagal)
# Skip cell ini jika Cell 3 berhasil

# Error umum dan solusinya:
# 1. 'ninja not found' → !apt-get install ninja-build -y
# 2. 'CUDA version mismatch' → pastikan torch.version.cuda cocok dengan nvcc --version
# 3. 'gcc: error' → !apt-get install build-essential -y
# 4. 'No module named torch' during setup.py → pastikan torch sudah terinstall SEBELUM compile

# Jalankan diagnostic:
print('=== DIAGNOSTIC ===')
!nvcc --version 2>/dev/null || echo 'nvcc tidak ditemukan'
!ninja --version 2>/dev/null || echo 'ninja tidak ditemukan — install: apt-get install ninja-build'
import torch
print(f'torch.version.cuda: {torch.version.cuda}')
print(f'CUDA available    : {torch.cuda.is_available()}')
print(f'Compiler path     : ', end='')
!which gcc

In [ ]:
# Cell 5: Custom COCO Keypoint dataset untuk DETR
# Deformable DETR menggunakan dataset class yang berbeda dari MMPose.
# Kita buat wrapper sederhana di atas COCO format.

import torch
from torch.utils.data import Dataset
from pycocotools.coco import COCO
import numpy as np
import cv2
from torchvision import transforms

class SoccerKeypointDataset(Dataset):
    """
    Dataset class untuk Deformable DETR pada soccer field keypoint detection.
    
    Setiap sample: {image_tensor, target_dict}
    target_dict berisi:
        - keypoints: (N_kpt*3,) = [x1,y1,v1, x2,y2,v2, ...] normalized
        - boxes    : (1, 4) = [cx, cy, w, h] normalized (full-image bbox)
        - labels   : (1,) = class index
        - image_id : int
    """
    def __init__(self, ann_file, images_dir, image_size=800, n_keypoints=32):
        self.coco        = COCO(ann_file)
        self.images_dir  = images_dir
        self.image_size  = image_size
        self.n_keypoints = n_keypoints
        self.img_ids     = list(self.coco.imgs.keys())
        self.transform   = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
    
    def __len__(self): return len(self.img_ids)
    
    def __getitem__(self, idx):
        img_id   = self.img_ids[idx]
        img_info = self.coco.imgs[img_id]
        img_path = os.path.join(self.images_dir, img_info['file_name'])
        
        img = cv2.imread(img_path)
        if img is None:
            img = np.zeros((480, 640, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        # Resize
        img_resized = cv2.resize(img, (self.image_size, self.image_size))
        img_tensor  = self.transform(img_resized)
        
        # Annotations
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns    = self.coco.loadAnns(ann_ids)
        
        if anns:
            ann = anns[0]  # Ambil annotation pertama (single instance)
            kpts_raw = np.array(ann['keypoints'], dtype=np.float32).reshape(-1, 3)
            # Normalize ke [0,1]
            kpts_norm = kpts_raw.copy()
            kpts_norm[:, 0] /= w
            kpts_norm[:, 1] /= h
            target_kpts = torch.tensor(kpts_norm.flatten(), dtype=torch.float32)
        else:
            target_kpts = torch.zeros(self.n_keypoints * 3, dtype=torch.float32)
        
        # Full-image bounding box normalized [cx, cy, w, h]
        target_box = torch.tensor([0.5, 0.5, 1.0, 1.0], dtype=torch.float32)
        
        target = {
            'keypoints': target_kpts,
            'boxes'    : target_box.unsqueeze(0),
            'labels'   : torch.zeros(1, dtype=torch.long),
            'image_id' : torch.tensor(img_id),
            'orig_size': torch.tensor([h, w]),
        }
        return img_tensor, target

# Test dataset
ds_train = SoccerKeypointDataset(
    ann_file=f'{COCO_ANN_DIR}/train.json',
    images_dir=f'{IMAGES_DIR}/train',
)
print(f'Train dataset: {len(ds_train)} samples')
img_t, target_t = ds_train[0]
print(f'Image tensor : {img_t.shape}')
print(f'Keypoints    : {target_t["keypoints"].shape}')

In [ ]:
# Cell 6: Build Deformable DETR model
import sys
sys.path.insert(0, DETR_DIR)

import argparse
from models import build_model

def build_detr_model(backbone='resnet50', n_keypoints=32, device='cuda'):
    """
    Build Deformable DETR model.
    Backbone: 'resnet50' atau 'resnet101'
    """
    args = argparse.Namespace(
        # Model architecture
        backbone=backbone,
        dilation=False,
        position_embedding='sine',
        num_feature_levels=4,
        # Transformer
        enc_layers=6, dec_layers=6,
        dim_feedforward=1024, hidden_dim=256,
        dropout=0.1, nheads=8,
        num_queries=1,         # Soccer field = single instance
        dec_n_points=4, enc_n_points=4,
        two_stage=False,
        # Loss
        aux_loss=True,
        set_cost_class=2, set_cost_bbox=5, set_cost_giou=2,
        cls_loss_coef=2, bbox_loss_coef=5, giou_loss_coef=2,
        eos_coef=0.1,
        # Dataset
        num_classes=1,         # Hanya satu class: soccer_field
        masks=False,
        device=device,
        # Keypoint extension
        num_keypoints=n_keypoints,
    )
    model, criterion, postprocessors = build_model(args)
    return model, criterion, postprocessors, args

model, criterion, postprocessors, args = build_detr_model(
    backbone='resnet50', n_keypoints=32
)
model = model.cuda()

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model built: {n_params/1e6:.1f}M trainable parameters')
print('DETR R50 ready')

In [ ]:
# Cell 7: Training loop Deformable DETR
# ┌──────────────────────────────────────────────────────┐
BACKBONE  = 'resnet50'   # ganti: 'resnet50' | 'resnet101'
VARIANT   = 'r50'        # ganti: 'r50'      | 'r101'
EPOCHS    = 50           # DETR konvergen lebih cepat dari CNN
BATCH     = 4            # Kurangi ke 2 jika OOM
# └──────────────────────────────────────────────────────┘

from torch.utils.data import DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from tqdm import tqdm

output_dir = f'{PROJECT_ROOT}/artifacts/weights/detr/{VARIANT}'
os.makedirs(output_dir, exist_ok=True)

# Datasets + DataLoaders
ds_train = SoccerKeypointDataset(f'{COCO_ANN_DIR}/train.json', f'{IMAGES_DIR}/train')
ds_val   = SoccerKeypointDataset(f'{COCO_ANN_DIR}/val.json',   f'{IMAGES_DIR}/val')

dl_train = DataLoader(ds_train, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
dl_val   = DataLoader(ds_val,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

# Model
model, criterion, postprocessors, args_detr = build_detr_model(BACKBONE, 32)
model = model.cuda()

# Optimizer: berbeda lr untuk backbone vs head
param_dicts = [
    {'params': [p for n,p in model.named_parameters() if 'backbone' not in n and p.requires_grad]},
    {'params': [p for n,p in model.named_parameters() if 'backbone'     in n and p.requires_grad],
     'lr': 2e-5},   # backbone lr = 10× lebih kecil
]
optimizer = optim.AdamW(param_dicts, lr=2e-4, weight_decay=1e-4)
scheduler = StepLR(optimizer, step_size=40, gamma=0.1)

best_val_loss = float('inf')
t_start = datetime.now()

for epoch in range(EPOCHS):
    # Training
    model.train()
    criterion.train()
    train_losses = []
    
    for imgs, targets in tqdm(dl_train, desc=f'Epoch {epoch+1}/{EPOCHS}', leave=False):
        imgs = imgs.cuda()
        targets = [{k: v.cuda() if isinstance(v, torch.Tensor) else v
                    for k, v in t.items()} for t in targets]
        
        outputs   = model(imgs)
        loss_dict = criterion(outputs, targets)
        weight_dict = criterion.weight_dict
        losses    = sum(loss_dict[k] * weight_dict[k]
                        for k in loss_dict.keys() if k in weight_dict)
        
        optimizer.zero_grad()
        losses.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)  # Gradient clipping
        optimizer.step()
        train_losses.append(losses.item())
    
    # Validation
    model.eval()
    criterion.eval()
    val_losses = []
    with torch.no_grad():
        for imgs, targets in dl_val:
            imgs = imgs.cuda()
            targets = [{k: v.cuda() if isinstance(v, torch.Tensor) else v
                        for k, v in t.items()} for t in targets]
            outputs   = model(imgs)
            loss_dict = criterion(outputs, targets)
            losses    = sum(loss_dict[k] * weight_dict[k]
                            for k in loss_dict.keys() if k in weight_dict)
            val_losses.append(losses.item())
    
    train_loss = np.mean(train_losses)
    val_loss   = np.mean(val_losses)
    scheduler.step()
    
    print(f'  Epoch {epoch+1:3d}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}')
    
    # Simpan best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'val_loss': val_loss,
            'args': args_detr,
        }, f'{output_dir}/checkpoint_best.pth')
    
    # Periodic checkpoint setiap 10 epoch
    if (epoch + 1) % 10 == 0:
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'val_loss': val_loss,
        }, f'{output_dir}/checkpoint_epoch_{epoch+1}.pth')

duration = (datetime.now() - t_start).total_seconds() / 3600
log = {
    'model_family': 'detr', 'model_variant': VARIANT,
    'trained_at': t_start.isoformat(), 'duration_hours': round(duration, 2),
    'final_val_loss': float(best_val_loss),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'epochs': EPOCHS, 'batch_size': BATCH,
}
with open(f'{output_dir}/train_log.json', 'w') as f:
    json.dump(log, f, indent=2)

print(f'\nSelesai! Duration: {duration:.1f}h')
print(f'Best val loss: {best_val_loss:.4f}')
del model; gc.collect(); torch.cuda.empty_cache()
print('GPU memory cleared')

In [ ]:
# Cell 8: Verifikasi semua DETR weights
import glob as gl
print('Status DETR weights:')
for v in ['r50','r101']:
    d = f'{PROJECT_ROOT}/artifacts/weights/detr/{v}'
    best = gl.glob(f'{d}/checkpoint_best.pth')
    if best:
        size_mb = os.path.getsize(best[0]) / 1024**2
        with open(f'{d}/train_log.json') as f:
            lg = json.load(f)
        print(f'  DETR-{v.upper()}: {size_mb:.0f} MB | val_loss={lg["final_val_loss"]:.4f} | {lg["duration_hours"]:.1f}h')
    else:
        print(f'  DETR-{v.upper()}: belum selesai training')
print('\nSemua model sudah ditraining!')
print('Langkah selanjutnya: 03_evaluation.ipynb')